# Log-Scale Value-Floor Analysis

**What this analysis does:** plots any coin's full daily history on a
**log10(price) vs date** axis and fits a **pseudo value-floor line** through the
major cycle bottoms. It then shows how far the current price is from that floor
(overshoot/undershoot %) and projects where the floor sits over the next year —
the "dumps bounce off the floor" readout.

- **Series 1** = actual daily close price
- **Series 2** = pseudo value-floor (log-linear fit through cycle troughs)

**Any asset, either exchange:**
- Binance spot (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`) — no key needed
- OKX spot (e.g. `OKB-USDT`, `BTC-USDT`) — no key needed

> Tweak `CELL 1 · CONFIG` (symbol, exchange, anchors, tick step, horizon) and
> re-run all cells. Illustrative trace, **not financial advice, not a signal**.

In [1]:
# ─── CELL 1 · CONFIG — tweak these ────────────────────────────────
EXCHANGE        = "binance"      # "binance" (spot) or "okx" (spot)
SYMBOL          = "BTCUSDT"      # binance: BTCUSDT / okx: OKB-USDT
BAR             = "1d"           # binance "1d"; okx "1D"
LIMIT_FROM      = None           # include bars from this date (None = all history)
FLOOR_ANCHORS   = [              # cycle-bottom anchors: (yyyy-mm-dd, label)
    ("2018-12-15", "2018 bottom"),
    ("2020-03-13", "COVID bottom"),
    ("2022-11-21", "2022 bottom"),
]
ANCHOR_WINDOW   = 40             # +/- days around each anchor to find the trough low
Y_TICK_STEP     = 10_000         # y-axis tick spacing in USD
PLOT_DAYS_HORIZON = 365          # project the floor line this many days forward
OUT_PNG         = "value_floor_chart.png"  # chart filename (goes into reports/_generated/)
SHOW            = False          # True = pop-up window, False = headless PNG
# ────────────────────────────────────────────────────────

In [2]:
# ─── CELL 2 · Imports — run once ─────────────────────────────────
import sys
from pathlib import Path
if hasattr(sys.stdout, "reconfigure"):
    try: sys.stdout.reconfigure(encoding="utf-8")
    except Exception: pass

import time
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg" if not SHOW else "TkAgg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, FixedLocator

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
print(f"✓ imports ready  · repo root = {ROOT}")

✓ imports ready  · repo root = C:\Users\Jerome\ai-trading-analyst


In [3]:
# ─── CELL 3 · Fetch daily history (Binance or OKX) ───────────────
def fetch_binance(symbol, interval):
    base = "https://api.binance.com/api/v3/klines"
    rows, start = [], 0
    while True:
        params = {"symbol": symbol, "interval": interval,
                  "limit": 1000, "startTime": start}
        r = requests.get(base, params=params, timeout=30)
        r.raise_for_status()
        batch = r.json()
        if not batch: break
        rows.extend(batch)
        if len(batch) < 1000: break
        start = batch[-1][0] + 1
        time.sleep(0.2)
    df = pd.DataFrame(rows, columns=["ts","o","h","l","c","v","ct","qv","t",
                                     "tb","tq","ig"])
    df["date"] = pd.to_datetime(df["ts"].astype(float), unit="ms")
    df["close"] = df["c"].astype(float)
    df["low"] = df["l"].astype(float)
    return df[["date","close","low"]].sort_values("date").drop_duplicates("date").reset_index(drop=True)

def fetch_okx(symbol, bar):
    rows, after = [], ""
    while True:
        params = {"instId": symbol, "bar": bar, "limit": "300"}
        if after: params["after"] = after
        r = requests.get("https://www.okx.com/api/v5/market/history-candles",
                         params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if data.get("code") != "0" or not data.get("data"): break
        cand = data["data"]
        rows.extend(cand)
        if len(cand) < 300: break
        after = cand[-1][0]
        if len(rows) > 4000: break
    df = pd.DataFrame(rows, columns=["ts","o","h","l","c","vol",
                                     "volCcy","volQuote","confirm"])
    df["date"] = pd.to_datetime(df["ts"].astype(int), unit="ms")
    df["close"] = df["c"].astype(float)
    df["low"] = df["l"].astype(float)
    return df[["date","close","low"]].sort_values("date").drop_duplicates("date").reset_index(drop=True)

fetch = {"binance": fetch_binance, "okx": fetch_okx}[EXCHANGE]
bar = BAR if EXCHANGE == "okx" else BAR
df = fetch(SYMBOL, bar)
if LIMIT_FROM:
    df = df[df["date"] >= pd.Timestamp(LIMIT_FROM)].reset_index(drop=True)
print(f"✓ {len(df)} {bar} bars  ({df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()})")

✓ 3322 1d bars  (2017-08-17 → 2026-09-20)


In [4]:
# ─── CELL 4 · Fit the pseudo value-floor line ───────────────────
def fit_floor(df, anchors, window_days):
    start = df["date"].iloc[0]
    pts = []
    for anchor, _label in anchors:
        a = pd.Timestamp(anchor)
        if a < start: continue
        win = df[(df['date'] >= a - pd.Timedelta(days=window_days)) &
                 (df['date'] <= a + pd.Timedelta(days=window_days))]
        if win.empty: continue
        trough = win.loc[win["low"].idxmin()]
        pts.append((trough["date"], trough["low"]))
    x = np.array([(d - start).days for d, _ in pts], float)
    y = np.array([np.log10(p) for _, p in pts], float)
    slope, intercept = np.polyfit(x, y, 1)
    return slope, intercept, pts

slope, intercept, floor_pts = fit_floor(df, FLOOR_ANCHORS, ANCHOR_WINDOW)
start = df["date"].iloc[0]
days = np.array((df["date"] - start).dt.days, float)
log_floor = np.asarray(slope * days + intercept, dtype=float)
print(f"✓ floor fit through {len(floor_pts)} troughs: "
      f"log10(price) = {slope:.6f}·days + {intercept:.4f}")
for d, p in floor_pts:
    print(f"    {d.date()} → ${p:,.2f}")

✓ floor fit through 3 troughs: log10(price) = 0.000503·days + 3.1942
    2018-12-15 → $3,156.26
    2020-03-13 → $3,782.13
    2022-11-21 → $15,476.00


In [5]:
# ─── CELL 5 · Chart: log10 price vs date ─────────────────────────
fig, ax = plt.subplots(figsize=(14, 7), dpi=120)

ax.semilogy(df["date"], df["close"], lw=1.4, color="#f7931a",
            label=f"{SYMBOL} close (log10 scale)")
floor = 10 ** log_floor
ax.plot(df["date"], floor, lw=2.0, ls="--", color="#4a5cff",
        label="Pseudo value-floor (cycle-lows fit)")

# project the floor line forward into the future
future = pd.date_range(df["date"].iloc[-1],
                       periods=max(2, PLOT_DAYS_HORIZON // 5), freq="5D")
fut_days = np.array((future - start).days, float)
ax.plot(future, 10 ** (slope * fut_days + intercept), lw=2.0, ls=":",
        color="#4a5cff", alpha=0.7, label="Floor projection")

# annotate the cycle bottoms
for anchor, note in FLOOR_ANCHORS:
    ts = pd.Timestamp(anchor)
    if ts < df["date"].iloc[0]: continue
    idx = (df["date"] - ts).abs().idxmin()
    ax.annotate(note, xy=(df["date"][idx], df["close"][idx]),
                xytext=(0, 40), textcoords="offset points", ha="center",
                arrowprops=dict(arrowstyle="->", color="#444"),
                fontsize=8, color="#333")

# y-axis: log base 10 with manual ticks every Y_TICK_STEP
ax.set_yscale("log")
lo, hi = df["close"].min(), df["close"].max()
ticks = np.arange((int(lo / Y_TICK_STEP) + 1) * Y_TICK_STEP,
                  int(hi) + 1, Y_TICK_STEP)
ax.yaxis.set_major_locator(FixedLocator(ticks))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:,.0f}"))

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title(f"{SYMBOL} — log-scale value-floor analysis (price vs pseudo"
             f" value-floor)", fontsize=13, fontweight="bold")
ax.set_ylabel("Price (USD, log10)")
ax.set_xlabel("Date")
ax.grid(True, which="both", alpha=0.25)
ax.legend(loc="upper left")
fig.tight_layout()

OUT_PNG = Path(ROOT) / "reports" / "_generated" / OUT_PNG
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=120)
print(f"✓ chart saved → {OUT_PNG}")
if SHOW: plt.show()
plt.close(fig)

✓ chart saved → C:\Users\Jerome\ai-trading-analyst\reports\_generated\value_floor_chart.png


In [6]:
# ─── CELL 6 · Readout: price vs floor + future projection ─────────
start_ts = pd.Timestamp(start)
price_now = float(df["close"].iloc[-1])
floor_today = 10 ** float(log_floor[-1])
print(f"{SYMBOL} close today    : ${price_now:,.2f}")
print(f"Pseudo floor today : ${floor_today:,.2f}")
print(f"Price vs floor     : {(price_now / floor_today - 1) * 100:+,.0f}% above floor")
print("Future floor-line projection (illustrative bounce-back zone):")
for months in [0, 3, 6, 9, 12, 18, 24]:
    d = start_ts + pd.Timedelta(days=int((df["date"].iloc[-1] - start_ts).days + months * 30.4))
    lvl = 10 ** (slope * (d - start_ts).days + intercept)
    print(f"    {d.date()}  →  floor ≈ ${lvl:,.2f}")

BTCUSDT close today    : $80,458.00
Pseudo floor today : $73,394.86
Price vs floor     : +10% above floor
Future floor-line projection (illustrative bounce-back zone):
    2026-09-20  →  floor ≈ $73,394.86
    2026-12-20  →  floor ≈ $81,558.13
    2027-03-21  →  floor ≈ $90,629.34
    2027-06-20  →  floor ≈ $100,709.49
    2027-09-19  →  floor ≈ $111,910.79
    2028-03-20  →  floor ≈ $138,349.77
    2028-09-18  →  floor ≈ $170,836.87


## Interpretation

Same methodology on any symbol: log10 price, dashed line = least-squares fit
(in log space) through the asset's major lows.

**How to read the % above floor**
- **0 to +50%** — price hugging the floor band; historically the "buy the
  bounce-back" zone (BTC majors and mid-caps).
- **+100% or more** — price far extended above its floor; entry risk is that a
  correction reverts toward the band.
- **Negative (below floor)** — either a red flag that the fit is fragile
  (parabolic / exchange tokens / young assets) or a genuinely deep value print.
  Size anyway as if the floor might fail.
- The floor is a **tendency across cycles**, not a guarantee; it is drawn in
  log space and is very sensitive to the anchor choice. Change the anchors in
  CELL 1 and watch the line move.

**This is a charting exercise. Not financial advice.**